<a href="https://colab.research.google.com/github/Jaish19/NLP-IITD/blob/main/Transformer%20%26%20BERT/LLM_Text_playaround.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install transformers

## CONVERSATIONAL BOT

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-large")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-large")

# Let's chat for 5 lines
for step in range(5):
    # encode the new user input, add the eos_token and return a tensor in Pytorch
    new_user_input_ids = tokenizer.encode(input(">> User:") + tokenizer.eos_token, return_tensors='pt')

    # append the new user input tokens to the chat history
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if step > 0 else new_user_input_ids

    # generated a response while limiting the total chat history to 1000 tokens,
    chat_history_ids = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)

    # pretty print last ouput tokens from bot
    print("DialoGPT: {}".format(tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

>> User:Hi How are you?


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


DialoGPT: I'm good , you ?
>> User:Yah, I'm great, Thanks for asking


KeyboardInterrupt: 

## QA BOT

In [5]:
import os
os.environ['HF_TOKEN'] = 'API_TOKEN_HERE'

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],
)

answer = client.question_answering(
    question="What is trainer name?",
    context="Session Name: Agentic AI, Trainer: Jai, Date: June 11th",
    model="distilbert/distilbert-base-uncased-distilled-squad",
)

print(answer)

QuestionAnsweringOutputElement(answer='Jai', end=38, score=0.9623733758926392, start=35)


## TEXT SUMMARIZATION

In [6]:
from transformers import BertTokenizerFast, EncoderDecoderModel
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizerFast.from_pretrained('mrm8488/bert-small2bert-small-finetuned-cnn_daily_mail-summarization')
model = EncoderDecoderModel.from_pretrained('mrm8488/bert-small2bert-small-finetuned-cnn_daily_mail-summarization').to(device)

def generate_summary(text):
    # cut off at BERT max length 512
    inputs = tokenizer([text], padding="max_length", truncation=True, max_length=512, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)

    output = model.generate(input_ids, attention_mask=attention_mask)

    return tokenizer.decode(output[0], skip_special_tokens=True)

text = '''Narendra Damodardas Modi (born September 17, 1950) is the 14th and current Prime Minister of India, serving since May 2014.
The first Indian Prime Minister born after independence, he previously served as the Chief Minister of Gujarat from 2001 to 2014,
and holds the record for the longest-serving elected head of government in Indian history. Modi led the BJP to historic, absolute majorities in the 2014,
2019, and 2024 general elections, making him the second leader in India's history to secure three consecutive terms as Prime Minister. His tenure focuses
on economic liberalization, technology-driven governance, and large-scale social welfare programs, positioning India as a major global economic force.'''
generate_summary(text)


Loading weights:   0%|          | 0/187 [00:00<?, ?it/s]

'narendra damodardas modi is the 14th and current prime minister of india. modi led the bjp to historic, absolute majorities in the 2014, 2019, and 2024 general elections. he holds the record for the longest - serving elected head of government in indian history.'